# Weather Across the Season at Citi Field

**Goal:** Visualize how temperature, pressure, humidity, and wind speed vary by month at Citi Field (New York Mets) to understand seasonal weather patterns and their potential effects on run scoring.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load league-wide game data and filter to NYM home games
data = pd.read_csv('../../data/league_weather_2021_2025.csv')
nym = data[data['home_team'] == 'NYM'].copy()

# Parse game_date to extract month
nym['game_date'] = pd.to_datetime(nym['game_date'])
nym['month'] = nym['game_date'].dt.month
month_map = {3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
             7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct'}
nym['month_name'] = nym['month'].map(month_map)

months = sorted(nym['month'].unique())
month_names = [month_map.get(m, str(m)) for m in months]

print(f"Total NYM home games: {len(nym)}")
print(f"\nGames per month:")
print(nym.groupby('month_name').size().reindex([month_map[m] for m in months]))

In [ ]:
# ============================================================
# Weather Distributions by Month
# ============================================================

weather_vars = {
    'temp_f':   {'label': 'Temperature (\u00b0F)', 'color': '#d62728'},
    'pres':     {'label': 'Pressure (hPa)',        'color': '#9467bd'},
    'rhum':     {'label': 'Humidity (%)',           'color': '#1f77b4'},
    'wspd_mph': {'label': 'Wind Speed (mph)',       'color': '#2ca02c'},
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (col, info) in zip(axes.flat, weather_vars.items()):
    box_data = [nym.loc[nym['month'] == m, col].dropna().values for m in months]
    bp = ax.boxplot(box_data, patch_artist=True, labels=month_names,
                    medianprops=dict(color='black', linewidth=1.5))
    for patch in bp['boxes']:
        patch.set_facecolor(info['color'])
        patch.set_alpha(0.6)
    ax.set_xlabel('Month', fontsize=11)
    ax.set_ylabel(info['label'], fontsize=11)
    ax.set_title(info['label'], fontsize=13)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

fig.suptitle('Weather Distributions by Month \u2014 Citi Field', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Humidity: Day vs. Night Games by Month
# ============================================================

# Day games start before 5 PM, night games at 5 PM or later
nym['time_of_day'] = nym['start_hour'].apply(lambda h: 'Day' if h < 17 else 'Night')

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(months))
width = 0.35

for i, (label, color) in enumerate([('Day', '#ff9900'), ('Night', '#003366')]):
    subset = nym[nym['time_of_day'] == label]
    grouped = subset.groupby('month')['rhum']
    means = grouped.mean().reindex(months)
    sems = grouped.sem().reindex(months)
    offset = (i - 0.5) * width
    bars = ax.bar(x + offset, means, width, yerr=sems, capsize=4,
                  label=label, color=color, alpha=0.75, edgecolor='black', linewidth=0.5)
    for xi, m in zip(x, means):
        if pd.notna(m):
            ax.text(xi + offset, m + sems.reindex(months).iloc[xi] + 0.3,
                    f'{m:.1f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(month_names, fontsize=11)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Humidity (%)', fontsize=12)
ax.set_title('Average Humidity: Day vs. Night Games by Month — Citi Field', fontsize=13)
ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

print(f"\nGame counts by month and time of day:")
print(nym.groupby(['month_name', 'time_of_day']).size().unstack(fill_value=0)
      .reindex([month_map[m] for m in months]))

In [ ]:
# ============================================================
# Average Total Runs: Day vs. Night Games by Month
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(months))
width = 0.35

for i, (label, color) in enumerate([('Day', '#ff9900'), ('Night', '#003366')]):
    subset = nym[nym['time_of_day'] == label]
    grouped = subset.groupby('month')['total_runs']
    means = grouped.mean().reindex(months)
    sems = grouped.sem().reindex(months)
    offset = (i - 0.5) * width
    bars = ax.bar(x + offset, means, width, yerr=sems, capsize=4,
                  label=label, color=color, alpha=0.75, edgecolor='black', linewidth=0.5)
    for xi, m in zip(x, means):
        if pd.notna(m):
            ax.text(xi + offset, m + sems.reindex(months).iloc[xi] + 0.1,
                    f'{m:.1f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(month_names, fontsize=11)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Total Runs', fontsize=12)
ax.set_title('Average Total Runs: Day vs. Night Games by Month — Citi Field', fontsize=13)
ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Summary Statistics Table
# ============================================================

summary = nym.groupby('month').agg(
    games=('temp_f', 'size'),
    temp_mean=('temp_f', 'mean'),
    temp_std=('temp_f', 'std'),
    pres_mean=('pres', 'mean'),
    pres_std=('pres', 'std'),
    rhum_mean=('rhum', 'mean'),
    rhum_std=('rhum', 'std'),
    wspd_mean=('wspd_mph', 'mean'),
    wspd_std=('wspd_mph', 'std'),
).round(2)

summary.index = [month_map.get(m, str(m)) for m in summary.index]
summary.index.name = 'Month'
summary.columns = ['Games', 'Temp Mean (\u00b0F)', 'Temp Std',
                    'Pressure Mean (hPa)', 'Pressure Std',
                    'Humidity Mean (%)', 'Humidity Std',
                    'Wind Mean (mph)', 'Wind Std']
summary